# Cuántos barrios en Madrid?

In [69]:
import pandas as pd
import unicodedata as ud

In [70]:
# cargo el listado de zonas SER (por calle, barrio y distrito) con la capacidad de plazas de aparcamiento
df_plazas = pd.read_csv('../data/ser_madrid/calles_SER_2025.csv', encoding='latin1', sep=';')
len(df_plazas['barrio'].unique())

63

In [71]:
# cargo el listado de tickets de aparcamiento (con distrito y bario) para 2024
df_tickets = pd.read_csv('../data/ser_madrid/2024.csv')
len(df_tickets['barrio'].unique())

63

<div  style="color:yellow;">
<h5>63 barrios diferentes</h5>
</div>

In [72]:
# compruebo si los identificadores de los barrios son los mismos en los dos datasets

# Crear conjuntos de tuplas (ID, Nombre)
pares_tickets = set(df_tickets[['barrio']].drop_duplicates().apply(tuple, axis=1))
pares_plazas = set(df_plazas[['barrio']].drop_duplicates().apply(tuple, axis=1))

# Comprobar pares presentes en tickets pero no en plazas
pares_solo_en_tickets = pares_tickets - pares_plazas
print(f"Pares (ID, Nombre) en df_tickets pero no en df_plazas: {(pares_solo_en_tickets)}")

# Comprobar pares presentes en plazas pero no en tickets
pares_solo_en_plazas = pares_plazas - pares_tickets
print(f"Pares (ID, Nombre) en df_plazas pero no en df_tickets: {(pares_solo_en_plazas)}")

if not pares_solo_en_tickets and not pares_solo_en_plazas:
    print("Todos los pares (ID, Nombre) de barrio coinciden entre ambos DataFrames.")
else:
    print("---\n¡ALERTA! Hay pares (ID, Nombre) de barrio que no coinciden entre ambos DataFrames.")

Pares (ID, Nombre) en df_tickets pero no en df_plazas: {('CIUDAD JARDÍN',), ('MOSCARDÓ',), ('ARGUELLES',), ('HISPANOAMÉRICA',), ('NIÑO JESÚS',), ('PUERTA DEL ÁNGEL',), ('CONCEPCIÓN',), ('PACÍFICO',), ('LOS CÁRMENES',), ('PILAR',)}
Pares (ID, Nombre) en df_plazas pero no en df_tickets: {('CONCEPCION',), ('CARMENES',), ('PACIFICO',), ('ARGÜELLES',), ('PUERTA DEL ANGEL',), ('NIÑO JESUS',), ('CIUDAD JARDIN',), ('EL PILAR',), ('HISPANOAMERICA',), ('MOSCARDO',)}
---
¡ALERTA! Hay pares (ID, Nombre) de barrio que no coinciden entre ambos DataFrames.


<div  style="color:yellow;">
<h5>¡¡¡Los códigos y nombres de los barrios son diferentes en los datasets!!!</h5>
</div>

In [73]:
# Normalizamos para que coincidan los nombres en ambos datasets
def normalize_text(text):
    text = text.lower()
    text = ud.normalize(
        'NFKD',
        text).encode('ascii', 'ignore').decode('utf-8')
    return text

# Aplicar la normalización a la columna de nombres de barrio en ambos DataFrames
df_tickets['barrio'] = df_tickets['barrio'].apply(normalize_text)
df_plazas['barrio'] = df_plazas['barrio'].apply(normalize_text)

In [74]:
# aún hay diferencias. las corrijo manualmente.
df_tickets['barrio'].replace('los carmenes', 'carmenes', inplace=True)
df_plazas['barrio'].replace('el pilar', 'pilar', inplace=True)

C:\Users\xabi\AppData\Local\Temp\ipykernel_14048\807017501.py:2: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df_tickets['barrio'].replace('los carmenes', 'carmenes', inplace=True)
C:\Users\xabi\AppData\Local\Temp\ipykernel_14048\807017501.py:3: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves a

In [75]:
df_plazas.head(3)

,gis_x,gis_y,cod_distrito,distrito,cod_barrio,num_barrio,barrio,calle,numero_finca,color,bateria_linea,numero_plazas
0,439592.91,4473566.23,1,CENTRO,11,1,palacio,"AGUAS, CALLE, DE LAS",2,077214010 Verde,Línea,7
1,439569.61,4473598.22,1,CENTRO,11,1,palacio,"AGUAS, CALLE, DE LAS",8,077214010 Verde,Línea,4
2,439536.49,4473428.32,1,CENTRO,11,1,palacio,"AGUILA, CALLE, DEL",17,077214010 Verde,Línea,4


<h5 style="color:yellow">Guardo las plazas por barrio</h5>

In [76]:
# convierto df_plazas en el df con el calculo las plazas por barrio
df_plazas = df_plazas.groupby('barrio')['numero_plazas'].sum().reset_index()
df_plazas.head()

,barrio,numero_plazas
0,acacias,3520
1,adelfas,2336
2,almagro,2848
3,almenara,3525
4,almendrales,2564


In [77]:
# ¿Cuántas plazas en total?
df_plazas['numero_plazas'].sum()

np.int64(177625)

In [78]:
df_plazas.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 63 entries, 0 to 62
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   barrio         63 non-null     object
 1   numero_plazas  63 non-null     int64 
dtypes: int64(1), object(1)
memory usage: 1.1+ KB


<h5 style="color:yellow">Compruebo que no se escape ningún barrio</h5>

In [79]:
# añado una columna para indicar los barrios de df_tickets que están en df_plazas
barrios_existentes = df_plazas['barrio'].unique()
df_tickets['barrio_en_df_plazas'] = df_tickets['barrio'].isin(barrios_existentes)

In [82]:
df_tickets.head(3)

,matricula_parquimetro,fecha_operacion,fecha_inicio,fecha_fin,cod_distrito,distrito,cod_barrio,barrio,tipo_zona,distintivo,minutos_tique,importe_tique,barrio_en_df_plazas
0,ELPARKING,2024-01-13 12:45:10,2024-01-13 12:45:10,2024-01-13 13:50:10,2,ARGANZUELA,2,acacias,AZUL,ECO,65,"0,30",True
1,TELPARK,2024-03-18 14:40:12,2024-03-18 14:40:12,2024-03-18 16:41:12,4,SALAMANCA,2,goya,AZUL,C,121,"2,50",True
2,EASYPARK,2024-02-27 12:00:12,2024-02-27 12:00:00,2024-02-27 12:20:00,6,TETUAN,4,almenara,AZUL,C,20,"0,60",True


In [47]:
# hay algún barrio de df_tickets que no esté en el listado de barrios?
df_tickets[df_tickets['barrio_en_df_plazas'] == False].count()

matricula_parquimetro    0
fecha_operacion          0
fecha_inicio             0
fecha_fin                0
cod_distrito             0
distrito                 0
cod_barrio               0
barrio                   0
tipo_zona                0
distintivo               0
minutos_tique            0
importe_tique            0
barrio_en_df_plazas      0
dtype: int64

<h5 style="color:yellow">Guardo en un CSV las plazas por barrio</h5>

In [83]:
df_plazas

,barrio,numero_plazas
0,acacias,3520
1,adelfas,2336
2,almagro,2848
3,almenara,3525
4,almendrales,2564
...,...,...
58,universidad,1409
59,valdeacederas,2163
60,valdezarza,1641
61,vallehermoso,3165


In [84]:
df_plazas.to_csv('../data/ser_madrid/plazas_barrios.csv', index=False)